# Preparació de les dades

Modifica les dades perquè els algorismes de ML puguin aprendre correctament a partir d'elles.

### Imports

In [1]:
# Importa les biblioteques, funcions, objectes... necessaris

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

### Carrega el dataset

In [2]:
df = pd.read_csv('../data/house_pricing.csv')

# Eliminamos la columna duplicada por error tipográfico antes de la separación
if 'Electtrical' in df.columns:
    df = df.drop(columns=['Electtrical'])

print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")

Dataset cargado con 992 filas y 36 columnas.


## Selecció de dades

Després de l'exploració anterior, pots decidir utilitzar o no utilitzar alguns dels conjunts de dades.

Per a aquest exercici **no hi ha cap decisió a prendre, utilitzes l'únic conjunt de dades que tenim**.

In [3]:
# Dividimos el dataset según la columna 'Split'
df_labeled = df[df['Split'] == 'labeled'].copy()
df_leaderboard = df[df['Split'] == 'leaderboard'].copy()

# Definimos variables predictoras (X) y objetivo (y) para el conjunto etiquetado
X_labeled = df_labeled.drop(columns=['Split', 'Id', 'SalePrice'])
y_labeled = df_labeled['SalePrice']

# Definimos el conjunto de pruebas finales (Leaderboard)
X_leaderboard = df_leaderboard.drop(columns=['Split', 'Id', 'SalePrice'])

# Hacemos el split de Validación para evitar la fuga de datos (Data Leakage)
X_train, X_val, y_train, y_val = train_test_split(X_labeled, y_labeled, test_size=0.2, random_state=42)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_leaderboard: {X_leaderboard.shape}")

X_train: (635, 33) | X_val: (159, 33) | X_leaderboard: (198, 33)


## Neteja de dades

### Elimina les característiques innecessàries (si n'hi ha)

In [4]:
def neteja_pipeline(X):
    X_clean = X.copy()

    # 1. Eliminación de variables irrelevantes o con nulos masivos sin estructura (MiscFeature)
    if 'MiscFeature' in X_clean.columns:
        X_clean = X_clean.drop(columns=['MiscFeature'])

    # 2. Imputación lógica: Si el tipo/acabado de garaje es nulo, significa que la casa no tiene garaje
    columnas_garaje = ['GarageType', 'GarageFinish']
    for col in columnas_garaje:
        if col in X_clean.columns:
            X_clean[col] = X_clean[col].fillna('NoGarage')

    # Si no tiene garaje, el año de construcción del garaje se iguala al de la casa para mantener la coherencia
    if 'GarageYrBlt' in X_clean.columns:
        X_clean['GarageYrBlt'] = X_clean['GarageYrBlt'].fillna(X_clean['YearBuilt'])

    # 'Electrical' contiene nulos mínimos aislados; imputamos con el valor más frecuente (moda)
    if 'Electrical' in X_clean.columns:
        X_clean['Electrical'] = X_clean['Electrical'].fillna(X_clean['Electrical'].mode()[0])

    return X_clean

# Aplicamos la limpieza de forma independiente
X_train_clean = neteja_pipeline(X_train)
X_val_clean = neteja_pipeline(X_val)
X_leaderboard_clean = neteja_pipeline(X_leaderboard)

### Tracta els valors nuls o erronis (si n'hi ha)

### Tracta les files duplicades que siguin errors (si n'hi ha)

### Decideix què fer amb els outliers (si n'hi ha)

Normalment, durant aquest pas pots decidir eliminar o canviar alguns outliers. No obstant això, **per a aquest exercici no eliminis cap outlier**, deixa'ls tal com estan.

## Construcció de dades

Decideix si vols crear noves característiques a partir de les existents. Pots ser tan creatiu com vulguis.

In [5]:
# No es necesario integrar fuentes de datos externas para este ejercicio.
X_train_int = X_train_clean.copy()
X_val_int = X_val_clean.copy()
X_leaderboard_int = X_leaderboard_clean.copy()

## Integració de dades

Decideix si vols integrar dades d'altres fonts.

**Això no és necessari per als exercicis.**

## Enginyeria de característiques

### Codificació

Aplica les codificacions que consideris més apropiades per a les variables categòriques.

In [6]:
# Identificamos columnas categóricas remanentes
columnas_categoricas = X_train_int.select_dtypes(include=['object']).columns.tolist()

# Clasificación de categorías según la guía
cols_ordinales = ['LandSlope', 'Utilities', 'CentralAir']
cols_nominales = [col for col in columnas_categoricas if col not in cols_ordinales]

# --- 1. Codificación Ordinal ---
orden_landslope = ['Gtl', 'Mod', 'Sev']
orden_utilities = ['NoSeWa', 'NoSewr', 'AllPub']
orden_centralair = ['N', 'Y']

ordinal_enc = OrdinalEncoder(categories=[orden_landslope, orden_utilities, orden_centralair],
                             handle_unknown='use_encoded_value', unknown_value=-1)

X_train_int[cols_ordinales] = ordinal_enc.fit_transform(X_train_int[cols_ordinales])
X_val_int[cols_ordinales] = ordinal_enc.transform(X_val_int[cols_ordinales])
X_leaderboard_int[cols_ordinales] = ordinal_enc.transform(X_leaderboard_int[cols_ordinales])

# --- 2. Codificación One-Hot (Nominales) ---
onehot_enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
onehot_enc.fit(X_train_int[cols_nominales])

def aplicar_one_hot(X, cols, encoder):
    matrix = encoder.transform(X[cols])
    new_cols = encoder.get_feature_names_out(cols)
    df_encoded = pd.DataFrame(matrix, columns=new_cols, index=X.index)
    return pd.concat([X.drop(columns=cols), df_encoded], axis=1)

# Variables de salida correctamente mapeadas por conjunto
X_train_encoded = aplicar_one_hot(X_train_int, cols_nominales, onehot_enc)
X_val_encoded = aplicar_one_hot(X_val_int, cols_nominales, onehot_enc)
X_leaderboard_encoded = aplicar_one_hot(X_leaderboard_int, cols_nominales, onehot_enc)

### Binning

Aplica binning a algunes columnes si ho consideres apropiat.

In [7]:
def aplicar_binning(X):
    X_bin = X.copy()
    # Límites e identificadores numéricos de rangos para la antigüedad de construcción
    bins = [0, 1950, 1980, 2000, 2010, 2030]
    labels = [0, 1, 2, 3, 4]

    X_bin['YearBuilt_Binned'] = pd.cut(X_bin['YearBuilt'], bins=bins, labels=labels).astype(int)
    return X_bin

# CORRECCIÓN: Pasamos las variables individuales generadas en la celda de Codificación
X_train_binned = aplicar_binning(X_train_encoded)
X_val_binned = aplicar_binning(X_val_encoded)
X_leaderboard_binned = aplicar_binning(X_leaderboard_encoded)

### Correlacions altes

Decideix què fer amb les característiques que estan molt altament correlacionades, si n'hi ha.

**Construción de datos**
Construcció de característiques

In [8]:
def construir_caracteristiques(X):
    X_built = X.copy()

    # 1. Indicador temporal
    X_built['AnosDesdeRemod'] = 2026 - X_built['YearRemodAdd']

    # 2. Superficie geométrica total habitable
    X_built['Total_Habitable_SF'] = X_built['1stFlrSF'] + X_built['2ndFlrSF'] + X_built['TotalBsmtSF']

    # 3. Mapeo global de baños disponibles
    bsmt_baths = X_built['BsmtFullBath'].fillna(0)
    X_built['Total_Bathrooms'] = X_built['FullBath'] + bsmt_baths

    return X_built

# Ahora X_train_binned existe perfectamente en la memoria de ejecución
X_train_final = construir_caracteristiques(X_train_binned)
X_val_final = construir_caracteristiques(X_val_binned)
X_leaderboard_final = construir_caracteristiques(X_leaderboard_binned)

Formatar dades

In [9]:
# Verificación final de tipologías estructurales y limpieza residual de nulos
print(f"¿Quedan columnas de tipo objeto/texto en la matriz?: {any(X_train_final.dtypes == 'object')}")
print(f"¿Existen valores nulos residuales en el set final de entrenamiento?: {X_train_final.isna().sum().sum()}")

# Muestra de dimensiones estructuradas y listas para la fase de modelado de datos
print(f"\nFormato final listo de Entrenamiento (X_train_final): {X_train_final.shape}")
print(f"Formato final listo de Validación (X_val_final): {X_val_final.shape}")
print(f"Formato final listo de Leaderboard (X_leaderboard_final): {X_leaderboard_final.shape}")

¿Quedan columnas de tipo objeto/texto en la matriz?: False
¿Existen valores nulos residuales en el set final de entrenamiento?: 0

Formato final listo de Entrenamiento (X_train_final): (635, 72)
Formato final listo de Validación (X_val_final): (159, 72)
Formato final listo de Leaderboard (X_leaderboard_final): (198, 72)


**FASE FINAL**
Reconstrucción y exportación a CSV preparado

In [10]:
# 1. Recuperamos los IDs y las variables objetivo (Target) para cada subconjunto
X_train_final['Id'] = df_labeled.loc[X_train_final.index, 'Id']
X_train_final['SalePrice'] = y_labeled.loc[X_train_final.index]
X_train_final['Split'] = 'train'

X_val_final['Id'] = df_labeled.loc[X_val_final.index, 'Id']
X_val_final['SalePrice'] = y_labeled.loc[X_val_final.index]
X_val_final['Split'] = 'validation'

X_leaderboard_final['Id'] = df_leaderboard.loc[X_leaderboard_final.index, 'Id']
X_leaderboard_final['SalePrice'] = np.nan  # El conjunto test original no tiene etiquetas de precio
X_leaderboard_final['Split'] = 'leaderboard'

# 2. Alineamos el orden exacto de las columnas para evitar desajustes estructurales en la unión
columnas_ordenadas = X_train_final.columns.tolist()
X_val_final = X_val_final[columnas_ordenadas]
X_leaderboard_final = X_leaderboard_final[columnas_ordenadas]

# 3. Concatenamos verticalmente todos los segmentos procesados en un único DataFrame Maestro
df_house_pricing_preparado = pd.concat([X_train_final, X_val_final, X_leaderboard_final], axis=0).reset_index(drop=True)

# 4. Guardamos la nueva base de datos limpia, codificada y con ingeniería de variables a un CSV útil
output_filename = '../house_pricing_prepared.csv'
df_house_pricing_preparado.to_csv(output_filename, index=False)

print(f"¡Éxito! Nueva base de datos exportada correctamente como '{output_filename}'")
print(f"Estructura final del archivo unificado: {df_house_pricing_preparado.shape}")

¡Éxito! Nueva base de datos exportada correctamente como 'house_pricing_prepared.csv'
Estructura final del archivo unificado: (992, 75)
